# DINOv3 Paper-Style PCA Visualization

This notebook builds a paper-style DINOv3 PCA visualization pipeline from local DINOv3 repo and local `.pth` weights.

It is designed to produce a smoother, higher-resolution rendering than the official patch-grid demo.

In [ ]:
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from scipy import signal
from sklearn.decomposition import PCA

plt.rcParams['figure.dpi'] = 200

In [ ]:
# Update these paths before running.
DINOV3_LOCATION = "/Users/yp/Documents/GithubReponsitory/dinov3"
MODEL_NAME = "dinov3_vits16plus"
MODEL_WEIGHTS = "/Users/yp/Documents/GithubReponsitory/dinov3/weights/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth"
FG_CLASSIFIER_PATH = "/Users/yp/Documents/GithubReponsitory/dinov3/notebooks/fg_classifier.pkl"
IMAGE_PATH = "/absolute/path/to/your/image.png"
OUTPUT_DIR = Path("./paper_style_pca_outputs")

PATCH_SIZE = 16
IMAGE_SIZE = 896
LAYER_INDEX = -1
PCA_WHITEN = True
FG_THRESHOLD = 0.5
PERCENTILES = (1, 99)
MASK_POWER = 0.8

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

In [ ]:
model = torch.hub.load(
    repo_or_dir=DINOV3_LOCATION,
    model=MODEL_NAME,
    source="local",
    weights=MODEL_WEIGHTS,
)
model = model.to(device).eval()

with open(FG_CLASSIFIER_PATH, "rb") as file:
    fg_classifier = pickle.load(file)

print("Model and foreground classifier loaded.")

In [ ]:
def resize_transform(image: Image.Image, image_size: int = IMAGE_SIZE, patch_size: int = PATCH_SIZE) -> torch.Tensor:
    width, height = image.size
    h_patches = int(image_size / patch_size)
    w_patches = int((width * image_size) / (height * patch_size))
    resized = TF.resize(image, (h_patches * patch_size, w_patches * patch_size))
    return TF.to_tensor(resized)


def extract_patch_features(image_tensor: torch.Tensor, layer_index: int = LAYER_INDEX) -> tuple[torch.Tensor, tuple[int, int]]:
    image_norm = TF.normalize(image_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
    image_batch = image_norm.unsqueeze(0).to(device)
    with torch.inference_mode():
        feats = model.get_intermediate_layers(
            image_batch,
            n=[layer_index],
            reshape=True,
            norm=True,
        )
    feat_map = feats[0].squeeze(0).detach().cpu()
    channels, h_patches, w_patches = feat_map.shape
    patch_features = feat_map.view(channels, -1).permute(1, 0).contiguous()
    return patch_features, (h_patches, w_patches)


def robust_normalize(projected_patches: np.ndarray, foreground_mask: np.ndarray, percentiles: tuple[float, float]) -> np.ndarray:
    normalized = projected_patches.copy()
    for channel in range(3):
        channel_values = normalized[..., channel][foreground_mask]
        low, high = np.percentile(channel_values, percentiles)
        normalized[..., channel] = np.clip(normalized[..., channel], low, high)
        normalized[..., channel] = (normalized[..., channel] - low) / max(high - low, 1e-6)
    return normalized


def render_pca(projected_patches: np.ndarray, fg_score: torch.Tensor, image_tensor: torch.Tensor, mask_power: float = MASK_POWER) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    patch_rgb = torch.from_numpy(projected_patches).permute(2, 0, 1).unsqueeze(0).float()
    soft_mask = fg_score.unsqueeze(0).unsqueeze(0).float()
    target_size = image_tensor.shape[1:]

    fullres_rgb = F.interpolate(
        patch_rgb,
        size=target_size,
        mode="bicubic",
        align_corners=False,
    )[0].permute(1, 2, 0).clamp(0, 1).numpy()

    fullres_mask = F.interpolate(
        soft_mask,
        size=target_size,
        mode="bicubic",
        align_corners=False,
    )[0, 0].clamp(0, 1).numpy()

    pca_only = fullres_rgb * np.power(fullres_mask[..., None], mask_power)
    image_np = image_tensor.permute(1, 2, 0).numpy()
    overlay = image_np * (1.0 - fullres_mask[..., None]) + pca_only * fullres_mask[..., None]
    overlay = np.clip(overlay, 0.0, 1.0)
    return pca_only, overlay, fullres_mask


In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
image_tensor = resize_transform(image)
patch_features, (h_patches, w_patches) = extract_patch_features(image_tensor)

fg_score = fg_classifier.predict_proba(patch_features.numpy())[:, 1].reshape(h_patches, w_patches)
fg_score = torch.from_numpy(signal.medfilt2d(fg_score, kernel_size=3)).float()
foreground_mask = (fg_score.numpy() > FG_THRESHOLD)

fg_patches = patch_features.numpy()[foreground_mask.reshape(-1)]
print("Patch features:", patch_features.shape)
print("Foreground patches:", fg_patches.shape)

In [ ]:
pca = PCA(n_components=3, whiten=PCA_WHITEN)
pca.fit(fg_patches)

projected_patches = pca.transform(patch_features.numpy()).reshape(h_patches, w_patches, 3)
projected_patches = robust_normalize(projected_patches, foreground_mask, PERCENTILES)

pca_only, overlay, fullres_mask = render_pca(projected_patches, fg_score, image_tensor)
patch_grid = projected_patches * foreground_mask[..., None]

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.imsave(OUTPUT_DIR / "input_resized.png", image_tensor.permute(1, 2, 0).numpy())
plt.imsave(OUTPUT_DIR / "foreground_score.png", fullres_mask, cmap="gray")
plt.imsave(OUTPUT_DIR / "patch_grid_pca.png", patch_grid)
plt.imsave(OUTPUT_DIR / "paper_style_pca.png", pca_only)
plt.imsave(OUTPUT_DIR / "paper_style_overlay.png", overlay)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(image_tensor.permute(1, 2, 0))
axes[0].set_title("Input")
axes[1].imshow(patch_grid)
axes[1].set_title("Patch grid PCA")
axes[2].imshow(pca_only)
axes[2].set_title("Paper-style PCA")
axes[3].imshow(overlay)
axes[3].set_title("Overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Saved outputs to:", OUTPUT_DIR.resolve())